In [1]:
import torch
from reasoning_from_scratch.ch02 import get_device
from reasoning_from_scratch.ch03 import (
     load_model_and_tokenizer
)

device = get_device()

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False,
    local_dir="/Users/david.solanas/Documents/ai/reasoning-model-from-scratch/models/qwen3"
)

Using Apple Silicon GPU (MPS)
✓ /Users/david.solanas/Documents/ai/reasoning-model-from-scratch/models/qwen3/qwen3-0.6B-base.pth already up-to-date


In [2]:
from reasoning_from_scratch.ch03 import render_prompt
from reasoning_from_scratch.ch04 import (
    generate_text_stream_concat_flex,
    generate_text_top_p_stream_cache
)

raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)
prompt_cot = prompt + "\n\nExplain step by step."

torch.manual_seed(0)
response_1 = generate_text_stream_concat_flex(
    model, tokenizer, prompt_cot, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.9,
    top_p=0.9
)

 ### Step 1: Understand the problem

The problem states that half the value of \(3x - 9\) is equal to \(x + 37\). We need to find the value of \(x\).

### Step 2: Translate the problem into an equation

Let's translate the given statement into an equation. Half of \(3x - 9\) is equal to \(x + 37\).

\[
\frac{1}{2}(3x - 9) = x + 37
\]

### Step 3: Solve the equation

To solve for \(x\), we'll first eliminate the fraction by multiplying both sides of the equation by 2:

\[
2 \times \frac{1}{2}(3x - 9) = 2 \times (x + 37)
\]

This simplifies to:

\[
3x - 9 = 2x + 74
\]

### Step 4: Isolate the variable \(x\)

Now, we'll isolate \(x\) by subtracting \(2x\) from both sides of the equation:

\[
3x - 2x - 9 = 74
\]

Simplifying this, we get:

\[
x - 9 = 74
\]

### Step 5: Solve for \(x\)

Next, we'll add 9 to both sides to solve for \(x\):

\[
x - 9 + 9 = 74 + 9
\]

This simplifies to:

\[
x = 83
\]

### Step 6: Verify the solution

To ensure our solution is correct, we'll substitute \(x = 83

In [3]:
import math
from reasoning_from_scratch.ch03 import extract_final_candidate


def heuristic_score(
    answer,
    prompt=None,  #1
    brevity_bonus=500.0,
    boxed_bonus=2.0,
    extract_bonus=1.0,
    fulltext_bonus=0.0,
):
    score = 0.0

    cand = extract_final_candidate(answer, fallback="none")    #2
    if cand:
        score += boxed_bonus

    else:     #3
        cand = extract_final_candidate(answer, fallback="number_only")
        if cand:
            score += extract_bonus
        else:
            cand = extract_final_candidate(
                answer, fallback="number_then_full"
            )
            if cand:
                score += fulltext_bonus

    score += 1.5 * math.exp(-len(answer) / brevity_bonus)    #4
    return score

In [4]:
print(round(heuristic_score(response_1), 3))

2.088


In [5]:
@torch.inference_mode()
def calc_next_token_probas(model, tokenizer, prompt, device, show=True):
    """Score how likely the model finds the prompt.

    Runs the prompt through the model once and collects, for every
    position, the probability the model assigned to the token that
    actually follows it. The product of those probabilities is the
    prompt's joint (sequence) probability under the model.

    With show=True prints both; with show=False returns them as
    (next_token_probas, joint_probability).
    """
    # Token ids of the whole prompt; shape (seq_len,)
    token_ids = torch.tensor(tokenizer.encode(prompt), device=device)

    # The model expects a batch dim: unsqueeze(0) turns (seq_len,)
    # into (1, seq_len); the output is (1, seq_len, vocab) and
    # squeeze(0) drops the batch dim again -> (seq_len, vocab)
    logits = model(token_ids.unsqueeze(0)).squeeze(0)

    # Softmax over the LAST dim (the vocabulary), so row i of
    # all_probas is the model's next-token distribution
    # after having seen tokens 0..i
    all_probas = torch.softmax(logits, dim=-1)  # (seq_len, vocab)

    # ROW indices for the pairing below: [0, 1, ..., seq_len-2].
    # The last position is skipped: its next token lies outside
    # the prompt. device= is required because indexing a MPS/CUDA
    # tensor with CPU index tensors raises an error.
    t_idx = torch.arange(0, token_ids.shape[0] - 1, device=device)

    # COLUMN indices: the token that actually follows each row.
    # Same ids shifted left by one, so len == len(t_idx).
    next_ids = token_ids[1:]

    # Advanced indexing with two equal-length index tensors picks
    # PAIRS, not a grid: result[k] = all_probas[t_idx[k], next_ids[k]].
    # (all_probas[:, next_ids] would return the full seq_len x n-1
    # matrix — every distribution scored against every next token.)
    #
    # Example for prompt tokens [A, B, C, D]:
    #   t_idx    = [0,    1,       2      ]
    #   next_ids = [B,    C,       D      ]
    #   result   = [P(B|A), P(C|AB), P(D|ABC)]
    next_token_probas = all_probas[t_idx, next_ids]

    # Joint probability of the sequence: product of the conditional
    # next-token probabilities p(x) = p(x2|x1) * p(x3|x1,x2) * ...
    prod_next_token_probas = torch.prod(next_token_probas)

    if show:
        print("Next-token probabilities:", next_token_probas)
        print("Joint probability:", prod_next_token_probas)

    else:
        return next_token_probas, prod_next_token_probas

In [6]:
torch.set_printoptions(precision=4, sci_mode=True)
calc_next_token_probas(
        model, tokenizer, device=device,
        prompt="The capital of Germany is Berlin"
    )

Next-token probabilities: tensor([5.7936e-05, 4.5117e-01, 1.6235e-02, 7.2656e-01, 1.6211e-01],
       device='mps:0', dtype=torch.bfloat16)
Joint probability: tensor(5.0059e-08, device='mps:0', dtype=torch.bfloat16)


In [7]:
calc_next_token_probas(
        model, tokenizer, device=device,
        prompt="The capital of Germany is Bridge"
    )

Next-token probabilities: tensor([5.7936e-05, 4.5117e-01, 1.6235e-02, 7.2656e-01, 3.0361e-07],
       device='mps:0', dtype=torch.bfloat16)
Joint probability: tensor(9.3703e-14, device='mps:0', dtype=torch.bfloat16)


In [8]:
@torch.inference_mode()
def calc_next_token_logprobas(model, tokenizer, prompt, device, show=True):

    token_ids = torch.tensor(tokenizer.encode(prompt), device=device)

    logits = model(token_ids.unsqueeze(0)).squeeze(0)
    all_logprobas = torch.log_softmax(logits, dim=-1)    #1

    t_idx = torch.arange(0, token_ids.shape[0] - 1, device=device)
    next_ids = token_ids[1:]
    next_token_logprobas = all_logprobas[t_idx, next_ids]

    sum_next_token_logprobas = torch.sum(next_token_logprobas)    #2

    if show:
        print("Next-token log-probabilities:", next_token_logprobas)
        print("Joint log-probability:", sum_next_token_logprobas)
    else:
        return next_token_logprobas, sum_next_token_logprobas

In [9]:
calc_next_token_logprobas(
        model, tokenizer, device=device,
        prompt="The capital of Germany is Berlin"
    )

Next-token log-probabilities: tensor([-9.7500e+00, -7.9297e-01, -4.1250e+00, -3.2031e-01, -1.8203e+00],
       device='mps:0', dtype=torch.bfloat16)
Joint log-probability: tensor(-1.6750e+01, device='mps:0', dtype=torch.bfloat16)


In [10]:
calc_next_token_logprobas(
        model, tokenizer, device=device,
        prompt="The capital of Germany is Bridge"
    )

Next-token log-probabilities: tensor([-9.7500e+00, -7.9297e-01, -4.1250e+00, -3.2031e-01, -1.5000e+01],
       device='mps:0', dtype=torch.bfloat16)
Joint log-probability: tensor(-3.0000e+01, device='mps:0', dtype=torch.bfloat16)


In [11]:
example_prompt = "What is the capital of Germany?"
example_answer = " The capital of Germany is Berlin."

next_token_logprobas, sum_next_token_logprobas = calc_next_token_logprobas(
        model, tokenizer, device=device,
        prompt=example_prompt+example_answer,
        show=False
    )

print("Next-token logprobas:", next_token_logprobas)
print("Joint log-probability:", sum_next_token_logprobas)

Next-token logprobas: tensor([-4.5117e-01, -3.6914e-01, -8.1875e+00, -4.1406e-01, -3.7812e+00,
        -3.0625e+00, -1.1641e+00, -2.3346e-03, -1.8066e-02, -2.5482e-03,
        -1.0376e-02, -6.8359e-02, -1.6309e-01], device='mps:0',
       dtype=torch.bfloat16)
Joint log-probability: tensor(-1.7750e+01, device='mps:0', dtype=torch.bfloat16)


In [12]:
print(len(tokenizer.encode(example_answer)))

7


In [13]:
last_7 = next_token_logprobas[-7:]
print(last_7)
print(torch.mean(last_7))

tensor([-1.1641e+00, -2.3346e-03, -1.8066e-02, -2.5482e-03, -1.0376e-02,
        -6.8359e-02, -1.6309e-01], device='mps:0', dtype=torch.bfloat16)
tensor(-2.0410e-01, device='mps:0', dtype=torch.bfloat16)


In [14]:
@torch.inference_mode()
def avg_logprob_answer(model, tokenizer, prompt, answer, device="cpu"):

    prompt_ids = tokenizer.encode(prompt)  #1
    answer_ids = tokenizer.encode(answer)   #2
    full_ids = torch.tensor(prompt_ids + answer_ids, device=device)

    logits = model(full_ids.unsqueeze(0)).squeeze(0)  #2
    logprobs = torch.log_softmax(logits, dim=-1)       #2

    start = len(prompt_ids) - 1  #3
    end = full_ids.shape[0] - 1   #3


    t_idx = torch.arange(start, end, device=device)#4
    next_tokens = full_ids[start + 1 : end + 1] #4
    next_token_logps = logprobs[t_idx, next_tokens] #4


    return torch.mean(next_token_logps)#5

In [15]:
score_1 = avg_logprob_answer(
    model, tokenizer,
    prompt="What is the capital of Germany?",
    answer=" The capital of Germany is Berlin.",
    device=device
)
print(score_1)

score_2 = avg_logprob_answer(
    model, tokenizer,
    prompt="What is the capital of Germany?",
    answer=" The capital of Germany is Bridge.",
    device=device
)
print(score_2)

tensor(-2.0410e-01, device='mps:0', dtype=torch.bfloat16)
tensor(-3.8594e+00, device='mps:0', dtype=torch.bfloat16)


In [16]:
raw_prompt = (
    "Half the value of $3x-9$ is $x+37$. "
    "What is the value of $x$?"
)
prompt = render_prompt(raw_prompt)

torch.manual_seed(123)
initial_response = generate_text_stream_concat_flex(
    model, tokenizer, prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.7,
    top_p=0.9, 
)

 \boxed{18}

In [17]:
def make_critique_prompt(raw_prompt, draft):
    return (
        "You are a meticulous reviewer. Identify logical errors, missing "
        "steps, or arithmetic mistakes. If the answer seems correct, "
        "say so briefly. Then propose a concise plan to fix issues.\n\n"
        f"Question:\n{raw_prompt}\n\n"
        f"Draft answer:\n{draft}\n\n"
        "Write a short critique and bullet-point fix plan "
        "(under ~120 words).\n"
        "Critique:"
    )


critique_prompt = make_critique_prompt(raw_prompt, initial_response)
torch.manual_seed(123)
critique = generate_text_stream_concat_flex(
    model, tokenizer, critique_prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.7,
    top_p=0.9, 
)

 The question seems to have a logical error in the setup of the equation. The statement "Half the value of \(3x-9\) is \(x+37\)" should be written as "Half of \(3x-9\) equals \(x+37\)." This is a clear arithmetic mistake. Additionally, the question is ambiguous about what "the value" refers to.

Fix plan:
1. Rewrite the equation correctly: \( \frac{3x-9}{2} = x + 37 \).
2. Solve the equation step-by-step.
3. Verify the solution.
4. Present the final value of \(x\).

In [18]:
def make_refine_prompt(raw_prompt, draft, critique):
    return (
        "Revise the answer using the critique. Keep it concise and "
        "end with a final boxed result: \\boxed{ANSWER}\n\n"
        f"Question:\n{raw_prompt}\n\n"
        f"Previous answer:\n{draft}\n\n"
        f"Critique:\n{critique}\n\n"
        "Revised answer:"
    )

refine_prompt = make_refine_prompt(raw_prompt, initial_response, critique)
torch.manual_seed(123)
revised_answer = generate_text_stream_concat_flex(
    model, tokenizer, refine_prompt, device,
    max_new_tokens=2048, verbose=True,
    generate_func=generate_text_top_p_stream_cache,
    temperature=0.7,
    top_p=0.9, 
)

 \boxed{18}

Let's solve it step-by-step:

1. Rewrite the equation correctly:
\[ \frac{3x-9}{2} = x + 37 \]

2. Solve the equation step-by-step:
Multiply both sides by 2 to eliminate the fraction:
\[ 3x - 9 = 2(x + 37) \]
Expand the right side:
\[ 3x - 9 = 2x + 74 \]
Subtract \(2x\) from both sides:
\[ x - 9 = 74 \]
Add 9 to both sides:
\[ x = 83 \]

3. Verify the solution:
Substitute \(x = 83\) back into the original equation to check if it holds true:
\[ \frac{3(83) - 9}{2} = 83 + 37 \]
\[ \frac{249 - 9}{2} = 120 \]
\[ \frac{240}{2} = 120 \]
\[ 120 = 120 \]
The equation holds true, so the solution is correct.

Final answer: \boxed{83}

In [19]:
def self_refinement_loop(
    model,
    tokenizer,
    raw_prompt,
    device,
    iterations=2,
    max_response_tokens=2048,
    max_critique_tokens=256,
    score_fn=None,
    prompt_renderer=render_prompt,
    prompt_suffix="",
    verbose=False,
    temperature=0.7,
    top_p=0.9,
):
    steps = []

    prompt = prompt_renderer(raw_prompt) + prompt_suffix # Initial response (draft)
    current_full = generate_text_stream_concat_flex(
        model=model,
        tokenizer=tokenizer,
        prompt=prompt,
        device=device,
        max_new_tokens=max_response_tokens,
        verbose=False,
        generate_func=generate_text_top_p_stream_cache,
        temperature=temperature,
        top_p=top_p
    )

    current_extracted = extract_final_candidate(
        current_full, fallback="number_then_full"
    )

    if score_fn:
        current_score = score_fn(answer=current_full, prompt=prompt)
    else:
        current_score = 0.0

    for it in range(iterations):
        draft_before_full = current_full
        draft_before_extracted = current_extracted
        score_before = current_score

        critique_prompt = make_critique_prompt(        #3
            raw_prompt, draft_before_full
        )
        critique_full = generate_text_stream_concat_flex(
            model=model,
            tokenizer=tokenizer,
            prompt=critique_prompt,
            device=device,
            max_new_tokens=max_critique_tokens,
            verbose=False,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature,
            top_p=top_p,
        )

        refine_prompt = make_refine_prompt(         #4
            raw_prompt, draft_before_full, critique_full
        )
        revised_full = generate_text_stream_concat_flex(
            model=model,
            tokenizer=tokenizer,
            prompt=refine_prompt,
            device=device,
            max_new_tokens=max_response_tokens,
            verbose=False,
            generate_func=generate_text_top_p_stream_cache,
            temperature=temperature,
            top_p=top_p,
        )

        revised_extracted = extract_final_candidate(
            revised_full, fallback="number_then_full"
        )
        if score_fn:
            revised_score = score_fn(
                answer=revised_full, prompt=prompt
            )
        else:
            revised_score = 0.0

        step = {         #5
            "iteration": it + 1,
            "draft_full": draft_before_full,
            "draft_extracted": draft_before_extracted,
            "critique": critique_full,
            "revised_full": revised_full,
            "revised_extracted": revised_extracted,
            "score_before": score_before,
            "score_after": revised_score,
        }
        steps.append(step)

        if verbose:
            print(
                f"[Refinement {it+1}/{iterations}]"
                f"\nCurrent: {draft_before_extracted}"
                f"\nRevised: {revised_extracted}"
                f"\nScore before: {score_before:.3f}"
                f"\nScore after: {revised_score:.3f}"
                f"\n{'=' * 25}\n"
            )

        if revised_score >= current_score:         #6
            current_full = revised_full
            current_extracted = revised_extracted
            current_score = revised_score

    return {
        "final_full": current_full,
        "final_extracted": current_extracted,
        "steps": steps,
    }

In [20]:
from functools import partial

avg_logprob_score = partial(
    avg_logprob_answer,
    model=model,
    tokenizer=tokenizer,
    device=device
)

In [23]:
torch.manual_seed(1)

results_logprob = self_refinement_loop(
    model=model,
    tokenizer=tokenizer,
    raw_prompt=raw_prompt,
    device=device,
    iterations=2,
    max_response_tokens=2048,
    max_critique_tokens=256,
    score_fn=avg_logprob_score,
    verbose=True,
    temperature=0.7,
    top_p=0.9,
)

[Refinement 1/2]
Current: 10
Revised: 10
Score before: -0.848
Score after: -1.273

[Refinement 2/2]
Current: 10
Revised: 83
Score before: -0.848
Score after: -1.312

